# QSAR on rapamycin and its rapalog library

**A guided cheminformatics deliverable.** We take *rapamycin* (sirolimus) — the
macrocyclic mTOR inhibitor — describe it numerically, train a model on real mTOR
bioactivity data, sketch a small analog library (including the rapalogs already
in the clinic), predict their activity, and then **check the predictions against
real wet-lab measurements**.

### Read this first — what QSAR can and cannot do here

QSAR = *Quantitative Structure–Activity Relationship*: learn a function
`structure → activity` from many compounds, then predict activity for new ones.

Two honest reframings of the task, baked into the steps below:

1. **You cannot build a model from one molecule.** "Calculate rapamycin's
   descriptor" (Step 1) describes *one* point; a model (Step 2) needs *many*
   compounds with *measured* activity. So Step 2 trains on rapamycin's target
   (mTOR) bioactivity from ChEMBL, using the *same* descriptor we compute in Step 1.
2. **Rapamycin is a weird molecule for QSAR.** It is a 914 Da macrocycle that
   violates every "rule of 5" cutoff. Standard descriptors still work, but we must
   watch the **applicability domain** — is a new analog similar enough to the
   training data for the prediction to mean anything?

Every step has a **Rationale** and a **Caveat** block. The caveats are the point.

## Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, DataStructs
from rdkit.Chem import Draw, Descriptors, AllChem, rdFingerprintGenerator
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Draw import rdMolDraw2D  # noqa: F401
from chembl_webresource_client.new_client import new_client

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

RANDOM_SEED = 42
DATA = Path("../data"); DATA.mkdir(exist_ok=True)

# One Morgan fingerprint generator (ECFP4-equivalent) reused everywhere so the
# "descriptor" in Step 1 is identical to the model's features in Step 2.
MORGAN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def morgan(mol):
    "Return (bitvector-for-similarity, dense-uint8-array-for-sklearn)."
    bv = MORGAN.GetFingerprint(mol)
    arr = np.zeros((2048,), dtype=np.uint8); DataStructs.ConvertToNumpyArray(bv, arr)
    return bv, arr

def parent_smiles(smi):
    "Largest fragment (strip salts), canonicalized."
    m = Chem.MolFromSmiles(smi) if smi else None
    return Chem.MolToSmiles(rdMolStandardize.FragmentParent(m)) if m else None

print("RDKit ready.")

## Step 1 — Calculate the descriptor of rapamycin

**Rationale.** A *molecular descriptor* turns a structure into numbers a model can
use. Two families:

- **Physicochemical descriptors** (MW, LogP, H-bond donors/acceptors, TPSA…):
  few, interpretable, great for drug-likeness filters.
- **Structural fingerprints** (here **Morgan / ECFP4**, a 2048-bit vector): encode
  which substructures are present. These are what our QSAR model will actually
  learn from, so we compute rapamycin's fingerprint here and reuse it in Step 2.

We pull rapamycin's structure from ChEMBL (`CHEMBL413`) rather than typing a
914-atom SMILES by hand.

In [ ]:
mol_client = new_client.molecule
rapa_smiles = mol_client.filter(molecule_chembl_id="CHEMBL413").only(
    ["molecule_structures"])[0]["molecule_structures"]["canonical_smiles"]
rapamycin = Chem.MolFromSmiles(rapa_smiles)
Draw.MolToImage(rapamycin, size=(560, 420))

In [ ]:
# (a) Interpretable physicochemical descriptors + Rule-of-5 check
panel = {
    "MolWt": Descriptors.MolWt(rapamycin),
    "LogP":  Descriptors.MolLogP(rapamycin),
    "HBD":   Descriptors.NumHDonors(rapamycin),
    "HBA":   Descriptors.NumHAcceptors(rapamycin),
    "TPSA":  Descriptors.TPSA(rapamycin),
    "RotatableBonds": Descriptors.NumRotatableBonds(rapamycin),
    "Rings": Descriptors.RingCount(rapamycin),
    "HeavyAtoms": Descriptors.HeavyAtomCount(rapamycin),
}
ro5 = {"MolWt>500": panel["MolWt"] > 500, "LogP>5": panel["LogP"] > 5,
       "HBD>5": panel["HBD"] > 5, "HBA>10": panel["HBA"] > 10}
print("Physicochemical descriptors of rapamycin:")
for k, v in panel.items():
    print(f"  {k:14s} {v:8.2f}")
print("\nLipinski Rule-of-5 violations:", sum(ro5.values()), "->", {k: v for k, v in ro5.items() if v})

In [ ]:
# (b) The STRUCTURAL descriptor the model uses: a 2048-bit Morgan fingerprint
bv, rapa_fp = morgan(rapamycin)
print(f"Morgan/ECFP4 fingerprint: {rapa_fp.shape[0]} bits, {int(rapa_fp.sum())} of them 'on'.")
print("first 60 bits:", "".join(map(str, rapa_fp[:60])))

**Caveat.** Rapamycin violates **3 of 4** Rule-of-5 cutoffs (MW 914, LogP ~6,
HBA 13). It works anyway because it is a natural-product macrocycle that binds an
adaptor protein (FKBP12) — a reminder that "drug-likeness" rules describe typical
oral small molecules, not this chemical class. More importantly: **a descriptor of
a single molecule is not a model.** It says what rapamycin *is*, nothing about what
makes a compound *active*. For that we need many labelled compounds — Step 2.

## Step 2 — Build a mathematical model from known data

**Rationale.** Rapamycin's molecular target is **mTOR**. ChEMBL holds thousands of
compounds with measured mTOR inhibition (`CHEMBL2842`). We:

1. pull mTOR **IC50** values,
2. curate them (keep exact `=` values with a computed pChEMBL, strip salts, take
   the median per compound),
3. featurize each compound with the **same Morgan fingerprint** from Step 1, and
4. train a **random-forest regressor** to predict **pIC50** (= −log10 IC50; higher
   = more potent).

Predicting the continuous pIC50 (regression) rather than just active/inactive lets
us later compare *predicted potency* against *measured potency* head-to-head.

In [ ]:
# 1. Pull mTOR IC50 (cached to ../data so re-runs are instant)
cache = DATA / "mtor_ic50.parquet"
if cache.exists():
    raw = pd.read_parquet(cache)
else:
    act = new_client.activity
    recs = act.filter(target_chembl_id="CHEMBL2842", standard_type="IC50",
                      pchembl_value__isnull=False).only(
        ["molecule_chembl_id", "canonical_smiles", "pchembl_value", "standard_relation"])
    raw = pd.DataFrame.from_records(list(recs)); raw.to_parquet(cache)
print("raw mTOR IC50 rows:", len(raw))

# 2. Curate -> one median pIC50 per compound
raw["pchembl_value"] = pd.to_numeric(raw["pchembl_value"], errors="coerce")
clean = raw[(raw["standard_relation"] == "=") & raw["canonical_smiles"].notna()].dropna(subset=["pchembl_value"]).copy()
uniq = {s: parent_smiles(s) for s in clean["canonical_smiles"].unique()}
clean["parent"] = clean["canonical_smiles"].map(uniq)
clean = clean.dropna(subset=["parent"])
compounds = clean.groupby("parent", as_index=False)["pchembl_value"].median().rename(columns={"pchembl_value": "pIC50"})
print(f"unique compounds after curation: {len(compounds)}")
compounds["pIC50"].describe().round(2)

In [ ]:
# A quick look at the label distribution -- watch for imbalance/selection bias
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(compounds["pIC50"], bins=40, color="steelblue", edgecolor="white")
ax.axvline(6, color="red", ls="--", label="1 uM (pIC50=6)")
ax.set(xlabel="pIC50 (-log10 M)", ylabel="compounds", title="mTOR IC50 potency distribution (ChEMBL)")
ax.legend(); plt.show()
frac_potent = (compounds["pIC50"] >= 6).mean()
print(f"{frac_potent:.0%} of curated compounds are potent (pIC50 >= 6).")

> **Note the selection bias already visible:** the distribution is piled up at high
> potency. Medicinal-chemistry papers mostly report compounds they *optimised to be
> active*, so ChEMBL under-represents weak binders. Keep this in mind — it makes any
> "accuracy" look better than it would on a balanced screen.

In [ ]:
# 3-4. Featurize with Morgan fingerprints, train an RF regressor.
#      IMPORTANT: we will hold the rapalogs out (Step 4), so define them now and
#      EXCLUDE their exact structures from training to avoid a leaked test.
RAPALOGS = {  # name -> (ChEMBL id, short note)
    "Sirolimus (rapamycin)": ("CHEMBL413",     "parent macrolide; immunosuppressant"),
    "Everolimus":            ("CHEMBL1908360", "C40-O-(2-hydroxyethyl); oncology/transplant"),
    "Temsirolimus":          ("CHEMBL1201182", "C40 ester prodrug; renal cell carcinoma"),
    "Ridaforolimus":         ("CHEMBL2103839", "C40 phosphinate; investigational oncology"),
    "Zotarolimus":           ("CHEMBL219410",  "tetrazole at C40; drug-eluting stents"),
}
rapa_rows = {}
for name, (cid, note) in RAPALOGS.items():
    s = mol_client.filter(molecule_chembl_id=cid).only(["molecule_structures"])[0]["molecule_structures"]["canonical_smiles"]
    rapa_rows[name] = {"chembl_id": cid, "note": note, "smiles": s, "parent": parent_smiles(s)}
rapalog_parents = {r["parent"] for r in rapa_rows.values()}

train = compounds[~compounds["parent"].isin(rapalog_parents)].reset_index(drop=True)
train_bvs, X = [], []
for s in train["parent"]:
    bv, arr = morgan(Chem.MolFromSmiles(s)); train_bvs.append(bv); X.append(arr)
X = np.array(X); y = train["pIC50"].to_numpy()

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)
model = RandomForestRegressor(n_estimators=400, random_state=RANDOM_SEED, n_jobs=-1).fit(Xtr, ytr)
pred_te = model.predict(Xte)
print(f"training compounds (rapalogs excluded): {len(train)}")
print(f"held-out test:  R2 = {r2_score(yte, pred_te):.3f}   MAE = {mean_absolute_error(yte, pred_te):.2f} log units")

In [ ]:
# Predicted vs measured on the in-domain test set
fig, ax = plt.subplots(figsize=(4.8, 4.8))
ax.scatter(yte, pred_te, s=10, alpha=0.4)
lims = [min(yte.min(), pred_te.min()) - 0.3, max(yte.max(), pred_te.max()) + 0.3]
ax.plot(lims, lims, "k--", lw=1, label="perfect")
ax.set(xlabel="measured pIC50", ylabel="predicted pIC50",
       title="In-domain test (random split)", xlim=lims, ylim=lims); ax.legend()
plt.show()

**Caveat.** Four honest limits of this model:

1. **Imbalance / selection bias** (above): most training compounds are potent, so
   the model's baseline job is easy.
2. **Noisy ground truth.** We pooled IC50s from many labs/assays; the median pIC50
   for a given compound can still be off by a log unit (we'll see this bite
   temsirolimus in Step 4).
3. **Interpolation, not extrapolation.** A random forest averages training values,
   so it **cannot predict a potency higher than what it has largely seen** — it
   regresses toward the mean at the extremes.
4. **Leakage.** Several rapalogs are *already in ChEMBL's mTOR set*. Predicting a
   compound that sat in training is memorization, not prediction — which is exactly
   why we removed them above and will test them held-out in Step 4.

## Step 3 — Render rapamycin's analog library (naming the real ones)

**Rationale.** Every clinically used rapamycin analog ("rapalog") modifies the
**same position — the C40 hydroxyl** — which points into solvent and is *not*
part of the FKBP12/mTOR binding face. That is why these modifications tune
pharmacokinetics/formulation while keeping potency. First we render the **five real
rapalogs** (named), then we **enumerate a small hypothetical library** by
substituting rapamycin's hydroxyls.

In [ ]:
# The five REAL rapalogs already in research / clinical use
real_mols = [Chem.MolFromSmiles(r["smiles"]) for r in rapa_rows.values()]
real_legends = [f"{name}\n({r['note']})" for name, r in rapa_rows.items()]
Draw.MolsToGridImage(real_mols, legends=list(rapa_rows.keys()), molsPerRow=3, subImgSize=(300, 240))

In [ ]:
# Enumerate a hypothetical analog library: substitute rapamycin's -OH groups with
# a handful of R-groups (some mimic real rapalog chemistry). Illustrative only.
r_groups = {
    "O-methyl":            "[C:1][O:2]C",
    "O-(2-hydroxyethyl)":  "[C:1][O:2]CCO",          # everolimus-like
    "O-acetyl":            "[C:1][O:2]C(C)=O",
    "O-carboxymethyl":     "[C:1][O:2]CC(=O)O",      # temsirolimus-fragment-like
    "O-(dimethylaminoethyl)": "[C:1][O:2]CCN(C)C",
}
analogs = {}
for rg_name, smarts in r_groups.items():
    rxn = AllChem.ReactionFromSmarts(f"[C:1][OX2H:2]>>{smarts}")
    for prod in rxn.RunReactants((rapamycin,)):
        m = prod[0]
        try:
            Chem.SanitizeMol(m)
            smi = Chem.MolToSmiles(m)
            if smi not in analogs:
                analogs[smi] = f"hyp: {rg_name}"
        except Exception:
            pass
# keep a manageable, deterministic sample
analog_smiles = sorted(analogs)[:9]
analog_mols = [Chem.MolFromSmiles(s) for s in analog_smiles]
print(f"generated {len(analogs)} unique hypothetical analogs; showing {len(analog_mols)}")
Draw.MolsToGridImage(analog_mols, legends=[analogs[s] for s in analog_smiles],
                     molsPerRow=3, subImgSize=(300, 230))

**Caveat.** This enumeration is **illustrative, not synthetic guidance**. The
reaction substitutes *whichever* hydroxyl matches — rapamycin has several — so many
products modify positions the real rapalogs leave untouched. Real analog design
targets the **C40-OH specifically** and respects synthetic accessibility. Treat these
as "points in chemical space near rapamycin" for the model to score, not as proposed
drugs.

## Step 4 — Predict activity, and check it against wet-lab reality

**Rationale.** The real test of a QSAR: **hold the real rapalogs out of training**
(done in Step 2), predict their pIC50, and compare to their **measured** mTOR pIC50
from ChEMBL. We also report each analog's **applicability-domain** distance (max
Tanimoto similarity to the training set) — predictions on dissimilar compounds are
less trustworthy. Then we score the hypothetical analogs (purely prospective).

In [ ]:
measured = compounds.set_index("parent")["pIC50"]   # includes the rapalogs' real values

def predict_and_ad(smi):
    m = Chem.MolFromSmiles(smi); bv, arr = morgan(m)
    pred = model.predict(arr.reshape(1, -1))[0]
    ad = max(DataStructs.BulkTanimotoSimilarity(bv, train_bvs))  # nearest training neighbour
    return pred, ad

# --- real rapalogs: predicted vs measured (held-out) ---
rows = []
for name, r in rapa_rows.items():
    pred, ad = predict_and_ad(r["parent"])
    meas = measured.get(r["parent"], np.nan)
    rows.append({"analog": name, "type": "real rapalog", "pred_pIC50": round(pred, 2),
                 "measured_pIC50": round(meas, 2) if not np.isnan(meas) else None,
                 "error": round(pred - meas, 2) if not np.isnan(meas) else None,
                 "AD_maxTanimoto": round(ad, 2)})
real_df = pd.DataFrame(rows)
real_df

In [ ]:
# --- hypothetical analogs: prospective predictions (no wet-lab truth) ---
hyp_rows = []
for smi in analog_smiles:
    pred, ad = predict_and_ad(smi)
    hyp_rows.append({"analog": analogs[smi], "type": "hypothetical",
                     "pred_pIC50": round(pred, 2), "measured_pIC50": None, "error": None,
                     "AD_maxTanimoto": round(ad, 2)})
hyp_df = pd.DataFrame(hyp_rows)
hyp_df

In [ ]:
# Recapitulation plot: predicted vs measured for the real rapalogs
rr = real_df.dropna(subset=["measured_pIC50"])
fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.scatter(rr["measured_pIC50"], rr["pred_pIC50"], s=80, color="crimson", zorder=3)
for _, row in rr.iterrows():
    ax.annotate(row["analog"].split(" ")[0], (row["measured_pIC50"], row["pred_pIC50"]),
                fontsize=8, xytext=(4, 3), textcoords="offset points")
lims = [5, 9.5]
ax.plot(lims, lims, "k--", lw=1, label="perfect prediction")
ax.fill_between(lims, [l-1 for l in lims], [l+1 for l in lims], color="gray", alpha=0.12,
                label="+/- 1 log unit")
ax.set(xlabel="measured pIC50 (ChEMBL, wet-lab)", ylabel="predicted pIC50 (held-out QSAR)",
       title="Do predictions recapitulate wet-lab?", xlim=lims, ylim=lims); ax.legend(fontsize=8)
plt.show()

### What the numbers say

- **Qualitatively — yes, recapitulated.** Every real rapalog is predicted potent
  (pIC50 ≈ 6.3–7.6), and every real rapalog *is* potent in the lab. Held out of
  training, the model still places them correctly in "active" space, and their
  applicability-domain similarity (~0.8–0.9) confirms they are **inside** the
  region the model can speak to.
- **Quantitatively — it underpredicts the very best.** The most potent rapalogs
  (sirolimus, everolimus, zotarolimus, measured ≈ 8.3–8.8) are predicted ≈ 1 log
  unit *low*. This is the random forest's **regression-to-the-mean**: it averages
  training compounds and cannot extrapolate past the bulk of what it saw.
- **A ground-truth wrinkle.** Temsirolimus's *measured* value here is low (~5.8)
  for a known-potent prodrug — an artifact of pooling heterogeneous assays. When
  "truth" is noisy, judging the model against it is itself uncertain.
- **Ridaforolimus** has no ChEMBL mTOR IC50, so its prediction is genuinely
  prospective — the model still scores it potent, consistent with its known activity.

**Bottom line / the assay-validation parallel.** A fingerprint QSAR trained on
ChEMBL mTOR data **correctly classifies the rapalogs as active and ranks them in the
right neighbourhood**, but it is not a potency *ruler* at the extremes and inherits
its training data's biases and noise. Used as a **triage filter** ("is this analog
plausibly active, and is it in-domain?") it recapitulates reality; used as a
**precise oracle** it does not. Knowing which question your model can answer — the
same discipline as validating an assay before trusting its readout — is the whole
game.

### Reproduce

```bash
uv sync                                    # build the environment (Python 3.11)
uv run jupyter lab notebooks/rapamycin_qsar.ipynb
```
Data are pulled live from ChEMBL and cached to `../data/` (git-ignored).